# Pooling and receptive fields

**Learning objective:** See how pooling reduces spatial resolution and calculate how receptive fields expand through a stack.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:15:41.696524: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974941.711936    3850 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974941.716243    3850 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:15:43.448514: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
x=tf.reshape(tf.range(1,17,dtype=tf.float32),(1,4,4,1))
maxp=tf.keras.layers.MaxPooling2D(2)(x); avgp=tf.keras.layers.AveragePooling2D(2)(x)
print("input:\n",x.numpy()[0,:,:,0]); print("max pool:\n",maxp.numpy()[0,:,:,0]); print("average pool:\n",avgp.numpy()[0,:,:,0])


input:
 [[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]
 [13. 14. 15. 16.]]
max pool:
 [[ 6.  8.]
 [14. 16.]]
average pool:
 [[ 3.5  5.5]
 [11.5 13.5]]


In [3]:
layers=[("conv3",3,1),("pool2",2,2),("conv3",3,1),("pool2",2,2)]
rf=1; jump=1; rows=[]
for name,k,s in layers:
    rf=rf+(k-1)*jump; jump*=s; rows.append({"layer":name,"receptive_field":rf,"effective_jump":jump})
display(pd.DataFrame(rows))


,layer,receptive_field,effective_jump
0,conv3,3,1
1,pool2,4,2
2,conv3,8,2
3,pool2,10,4


Deeper units can represent larger spatial context because each layer's local neighborhood is built from neighborhoods in the preceding layer.
